# 05 · Sağlamlık testleri ve sızıntı (kopya) denetimi
**Sağlamlık:** test girdilerine gürültü, sensör ofseti, eksik örnek (son değeri tut) ve dosya ortasından başlama (geçmiş yok, soğuk başlangıç) uygulanır;
modeller yeniden eğitilmez. EKF: başlangıç senaryolarında SoC0 = 1.0 varsayılır (gerçek SoC bilinmez).
**Sızıntı denetimi:** her test penceresinin akım profiline en çok benzeyen eğitim/val penceresi (Pearson); benzerlik ↔ model hatası ilişkisi.

In [ ]:
import sys, os, json, time, warnings
import os; ROOT = os.path.abspath('..')  # repository root (run notebooks from the notebooks/ folder)
SMOKE = False                 # True: çok küçük hızlı deneme (kurulum testi)  False
sys.path.insert(0, ROOT)
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, torch
from data_utils import *
from models.experiment import *
from models import engine
exp = Experiment(ROOT, smoke=SMOKE)
exp.load_data()
print('Cihaz:', exp.dev, '| W =', exp.cfg['W'])

In [ ]:
from models.ekf import ECMParams, run_ekf
sel = exp.load_selected(); MODELS = [m for m in BASE_SPECS if m in sel]
ROB_SEEDS = exp.cfg['FINAL_SEEDS'][:3]
P = ECMParams(json.load(open(exp.p['configs'] / 'ecm_parametreleri.json')))
rob = []
for si, (scen, sc) in enumerate(ROBUST_SCENARIOS.items()):
    dfs = {k: perturb(d, sc, seed=1000 + si) for k, d in exp.te_dfs.items()}
    pad = 'start_soc' in sc
    for name in MODELS:
        for seed in ROB_SEEDS:
            m, rec = exp.load_model('final', name, seed)
            sets = exp.test_sets_from(dfs, rec['spec']['window'], pad)
            rows, _ = engine.evaluate(m, sets, dfs, exp.cfg)
            rob += [{'senaryo': scen, 'model': name, 'seed': seed, **r} for r in rows]
    for (t, c), d in dfs.items():
        soc0 = 1.0 if pad else d.soc.iloc[0]
        est = run_ekf(d, P, soc0); k0 = 0 if pad else exp.cfg['W']
        rob.append({'senaryo': scen, 'model': '2RC-ECM + EKF', 'seed': 0, 'temp': t, 'cycle': c, **engine.metrics(est[k0:], d.soc.values[k0:])})
    print(f'✅ {scen}')
rob = pd.DataFrame(rob); rob.to_csv(exp.p['excel'] / '05_saglamlik_satirlar.csv', index=False)
rob_t = (rob.groupby(['senaryo', 'model', 'seed']).rmse.mean() * 100).groupby(['senaryo', 'model']).mean().unstack('model').reindex(list(ROBUST_SCENARIOS))
print(rob_t.round(3).to_string())

In [ ]:
# Sızıntı denetimi: akım profili benzerliği (eğitim + val ↔ test), L = W
ref = {k: d for k, d in exp.role_dfs.items()}
aud = leakage_audit(ref, exp.te_dfs, L=exp.cfg['W'], device=exp.dev)
summ = aud.groupby(['temp', 'cycle']).agg(pencere=('max_corr', 'size'), medyan_max_corr=('max_corr', 'median'),
                                           oran_corr_090=('max_corr', lambda x: (x >= .90).mean()), oran_corr_095=('max_corr', lambda x: (x >= .95).mean()),
                                           oran_corr_099=('max_corr', lambda x: (x >= .99).mean()), medyan_mad_A=('mad_A', 'median'))
print(summ.round(3).to_string())
# hata ilişkisi: önerilen modelin en iyi seed serisinde pencere içi RMSE
recs = [r for r in exp.records('final') if r['name'] == 'Önerilen (RC-Hibrit)']
best = min(recs, key=lambda r: r['val_rmse_final']); _, _, pq = exp.paths('final', best['name'], best['seed'])
ser = load_table(pq)
L = exp.cfg['W']; errs = []
for (t, c), g in aud.groupby(['temp', 'cycle']):
    e = ser[(ser.temp == t) & (ser.cycle == c)].err.values
    for a in g.start.values:                       # pencere [a, a+L) ↔ seri satırı a (pencere sonu a+L−1, W = L)
        seg = e[a:a + 60]; errs.append(float(np.sqrt(np.mean(seg ** 2))) if len(seg) else np.nan)
aud['model_rmse_60s'] = errs
from scipy import stats
rho = aud.groupby('temp').apply(lambda g: stats.spearmanr(g.max_corr, g.model_rmse_60s, nan_policy='omit').correlation)
print('\nSpearman (benzerlik, model hatası) sıcaklık bazında — negatif ve büyük olsaydı ezber şüphesi:'); print(rho.round(3).to_string())
aud.to_csv(exp.p['excel'] / '05_sizinti_pencereler.csv', index=False)
with pd.ExcelWriter(exp.p['excel'] / '05_saglamlik_sizinti.xlsx') as xw:
    rob_t.to_excel(xw, sheet_name='Sağlamlık RMSE')
    summ.to_excel(xw, sheet_name='Sızıntı özet'); rho.rename('spearman').to_frame().to_excel(xw, sheet_name='Sızıntı-hata')